# Simple Linear Regression as a Neural Network

$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

$$
\hat{y}_i = b + w x_{i}
$$

$$
\hat{y}_i = \text{id}(b + w x_i)
$$

## Gradient Descent

$$
L(w) = (w - 1)^2 + 0.5
$$

$$
w_1 = w_0 - \eta \frac{dL}{dw}\big|_{w=w_0}
$$

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

In [ ]:
# Generate synthetic data to setup a linear regression problem as an example

# Number of observations
n = 100

# Number of predictors
p = 1

np.random.seed(23)

# A n x p matrix of predictors

X = np.random.randn(n, p)

# Random noise 
e = np.random.randn(n)

# True coefficient of x
beta = np.array([2])

# Response variable
y = 2 + X @ beta + e

# Standardize y for better numerical stability
# (x is drawn from N(0,1), so it is already approximately standardized)

y = (y - np.mean(y)) / np.std(y)

## Gradient Descent for the Simple Linear Regression

$$
\begin{align*}
w^{\text{new}} & = w^{\text{old}} - \eta \frac{\partial L_i}{\partial w} \\
b^{\text{new}} & = b^{\text{old}} - \eta \frac{\partial L_i}{\partial b}
\end{align*}
$$

$$
L = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

$$
\begin{align*}
\frac{\partial L}{\partial w} & = -\frac{2}{n} x^\top (y - \hat{y}) \\
\frac{\partial L}{\partial b} & = -\frac{2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)
\end{align*}
$$

## Simple Linear Regression as an ANN with Gradient Descent

In [ ]:
# Hyperparameters

# How many times to go through the data
epochs = 200

# The learning rate
learning_rate = 0.05

# First we initialize the parameters to small random values

w = np.random.randn(p, 1)  # shape (p, 1) so that X @ w works for any p 
b = np.random.randn(1)

print("Initial weights:", w)
print("Initial bias:", b)

losses = []

# Training loop
for epoch in range(epochs):
    # Forward pass
    y_pred = X @ w + b
    
    # Compute loss (MSE)
    loss = np.mean((y.reshape(-1, 1) - y_pred) ** 2)
    losses.append(loss)
    
    # Backward pass (compute gradients)
    errors = y.reshape(-1, 1) - y_pred

    gradient_w = -2 * X.T @ errors / n
    gradient_b = -2 * errors.mean()
    
    # Update parameters
    w = w - learning_rate * gradient_w
    b = b - learning_rate * gradient_b

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch + 1} Updated weights:", w)
        print(f"Epoch {epoch + 1} Updated bias:", b)

# Print final parameters
print("Weight:", w.item())
print("Bias:", b.item())

# Plot training loss
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs (NumPy)')
plt.show()

# Plot predictions
y_pred_final = X @ w + b

plt.scatter(X, y, label='Data')
plt.plot(X, y_pred_final, color='red', label='Prediction')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()

## Implementation in PyTorch

In [ ]:
import torch

# Convert the existing NumPy data to PyTorch tensors
X_torch = torch.from_numpy(X).float()
y_torch = torch.from_numpy(y.reshape(-1, 1)).float()

# Define a linear regression model: one output and no hidden layers
linear_model_torch = nn.Linear(p, 1)

criterion_torch = nn.MSELoss()
optimizer_torch = torch.optim.SGD(linear_model_torch.parameters(), lr=0.05)

epochs_torch = 1000
losses_torch = []

for epoch in range(epochs_torch):
    # Forward pass and loss
    y_pred_torch = linear_model_torch(X_torch)
    loss_torch = criterion_torch(y_pred_torch, y_torch)
    losses_torch.append(loss_torch.item())

    # Compute gradients and update parameters
    optimizer_torch.zero_grad()
    loss_torch.backward()
    optimizer_torch.step()

print("Intercept:", linear_model_torch.bias.item())
print("Coefficient:", linear_model_torch.weight.detach().numpy().ravel())
print("Final MSE:", losses_torch[-1])

plt.plot(losses_torch)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("PyTorch Linear Regression Training Loss")
plt.show()

## What's next